# ShellToolMiddleware
**It gives a LangChain agent a shell tool backed by one persistent shell session.**
* Instead of writing a custom subprocess tool, LangChain manages the shell lifecycle, working directory, startup/shutdown commands, execution limits.

#### 1. What can ShellToolMiddleware do?
An agent can use it to perform things like.
* Run shell commands: ls, pwd, git, python, pytest, etc.
* Inspect and modify files.
* Run scripts.
* Run tests and inspect failures.
* Build/compile projects.
* Execute development/automation commands.
* Keep state across commands because the shell session is persistent.
* Start with predefined setup commands.
* Clean up with predefined shutdown commands.
* Run inside Docker or another execution policy instead of directly on the host.
* Limit command execution time and output.
* Redact sensitive information from command output before it reaches the model.

```text
User: "Run the tests and fix the failing test."

Agent:
  shell("pwd")
      ↓
  shell("pytest")
      ↓
  shell("sed -n '1,200p' tests/test_api.py")
      ↓
  shell("...edit/fix...")
      ↓
  shell("pytest")
      ↓
  "All tests pass."
```

## Example Workflow
* The notebook creates a deliberately failing calculator project
* Gives a LangChain agent access to a persistent shell session, and asks the agent to diagnose and fix the failure.

```text
                Load dependencies and environment
                            |
                            v
                Create the shell_agent_workspace directory
                            |
                            v
                Write calculator.py with an intentional bug
                            |
                            v
                Write test_calculator.py
                            |
                            v
                Configure ShellToolMiddleware
                            |
                            v
                Create the LangChain agent
                            |
                            v
                Agent inspects the workspace
                            |
                            v
        |--------------->Run pytest
        |                    |
        |                    v
        |              Tests pass?
        |                /       \
        |              No         Yes
        |              |           |
        |              v           v
        |       Inspect failure   Summarize the fix
        ------  and edit          and test result
                calculator.py           |
                                        |
                                        |
                                        |
                                        v
                              Shell session shuts down

Persistent shell state is shared across inspection, testing, and editing commands.

<!-- #### 2. Why "persistent shell" matters
With a persistent session, Imagine the model does:
```sh
    cd /workspace/my-project
    export ENV=dev
    git status
    pytest
```

```text
┌───────────────────────────────────────┐
│          Shell Session                │
│                                       │
│ cd /workspace/my-project              │
│          ↓                            │
│ current directory = my-project        │
│                                       │
│ export ENV=dev                        │
│          ↓                            │
│ ENV = dev                             │
│                                       │
│ pytest                                │
│          ↓                            │
│ runs inside my-project with ENV=dev   │
└───────────────────────────────────────┘
```

* The cd and export state can therefore be used by subsequent commands.
* That's one of the major differences between this middleware and a simplistic tool that calls `subprocess.run()` independently for every command.
* LangChain explicitly describes the middleware as exposing a single long-lived shell session.

#### 3. Parameters
The constructor is essentially:
```python
    ShellToolMiddleware(
        workspace_root="/workspace/my-project",  # The directory where the shell session starts.
        *,
        startup_commands=["python -m venv .venv","source .venv/bin/activate", "pip install -r requirements.txt", "export APP_ENV=test"], # Commands executed when the shell session starts.
        shutdown_commands=["rm -rf .pytest_cache"], # Commands executed before the shell is shut down
        execution_policy=HostExecutionPolicy(
            command_timeout=30.0,  # Maximum time for a command
            startup_timeout=30.0,  # Maximum time to start the shell
            termination_timeout=10.0,  # Time allowed to terminate it
            max_output_lines=100,  # Limit returned output lines
            max_output_bytes=None,  # Limit returned output size
        ), # This determines where and how commands actually execute [DockerExecutionPolicy(image="python:3.11-slim")]
        redaction_rules=RedactionRule(pii_type="api_key",detector=r"sk-[a-zA-Z0-9]{32}"), # This controls what gets removed/redacted from command output before the output is sent back to the model.
        tool_description='Use this tool only for running tests and inspecting project files. Do not install packages or modify files outside /workspace',
        tool_name="shell", # ['shell','terminal']
        shell_command="/bin/bash",  # This controls which shell executable is used.
        env={"APP_ENV": "test"},  # Environment variables supplied to the shell:
    )
``` -->

In [1]:
from pathlib import Path

from langchain.agents import create_agent
from langchain.agents.middleware import ShellToolMiddleware
from langchain.agents.middleware import HostExecutionPolicy
from dotenv import load_dotenv
import os
from rich import print
import sys
sys.path.append('..')
from utils.helper import print_messages

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

#### 1. Create the agent workspace
* Define a `shell_agent_workspace` directory relative to the notebook and creates it if it does not already exist. 
* The directory is the working area where the agent will inspect and modify files.

In [2]:
workspace = Path.cwd() / 'shell_agent_workspace'
workspace.mkdir(exist_ok=True)

#### 2. Write the calculator implementation
* This cell creates `calculator.py` in the workspace. 
* It writes two functions, `add` and `multiply`; 
* `add` intentionally subtracts instead, creating the bug that the agent is expected to discover and fix.

In [3]:
(workspace / "calculator.py").write_text(
"""
def add(a, b):
    return a - b

def multiply(a, b):
    return a * b
""".strip()
)

69

#### 3. Write the calculator tests
* Creates `test_calculator.py` with tests for addition and multiplication. 
* The addition test expects `add(2, 3)` to return `5`, so it will expose the intentional subtraction bug.

In [4]:
(workspace / "test_calculator.py").write_text(
    """
from calculator import add, multiply

def test_add():
    assert add(2, 3) == 5

def test_multiply():
    assert multiply(2, 3) == 6
""".strip()
)

print("Workspace:", workspace)
print("Files:", [p.name for p in workspace.iterdir()])

Workspace: 
/Users/geetikabatra/genai_bootcamp/implementation/langchain-learning/notebooks/07_middlewares/shell_agent_workspace

Files:
['test_calculator.py', 'calculator.py']

### 4. Configure the shell middleware
* Configure a persistent shell session rooted at the workspace. 
* Define startup and shutdown commands, host execution limits, the tool name, and instructions that tell the agent how to inspect, test, and fix the project.

In [5]:
shell_middleware = ShellToolMiddleware(
    workspace_root=workspace,

    # For this notebook example, run directly on the host.
    # NOTE: The agent can execute arbitrary shell commands with the permissions of this Python process.
    execution_policy=HostExecutionPolicy(
        command_timeout=300,      # 5 minutes per shell command
        startup_timeout=60,       # shell startup
        termination_timeout=10,   # cleanup if command is killed
    ),
    shell_command=["/bin/zsh"],
    startup_commands=[
        "source /Users/geetikabatra/genai_bootcamp/implementation/langchain-learning/.venv/bin/activate"],

    tool_description="""
        You are working inside a Python project workspace.

        Use the shell tool to:
        - inspect files
        - run tests
        - inspect test failures
        - modify source files when necessary

        Your task is to diagnose and fix test failures.

        Rules:
        1. Inspect the project before changing anything.
        2. Run the tests.
        3. Understand the failure before making a change.
        4. Make the smallest appropriate fix.
        5. Run the tests again after the fix.
        6. Do not claim that tests pass unless you actually ran them.
        """,
)

#### 5. Create the LangChain agent
* It constructs a LangChain agent using `gpt-5-nano` and attaches the shell middleware. 
* The middleware supplies the shell tool automatically, so no separate tool list is needed for shell access.

In [6]:
agent = create_agent(
    model="openai:gpt-5.5",
    # ShellToolMiddleware automatically exposes a "shell" tool to the model.
    tools=[],
    middleware=[
        shell_middleware,
    ]
)

#### 6. Ask the agent to diagnose and fix the bug
Send the agent a structured task: 
  1. Inspect the workspace
  2. Run the tests
  3. Investigate any failure
  4. Fix the source
  5. Rerun the tests
  6. Summarize the result. 
The returned value is stored in `response`.

In [7]:
response = agent.invoke({
  "messages": [
    {
      "role": "user",
      "content": f"""You are working on a Python project.
        Workspace: {workspace}
        Please:
        1. Inspect the files.
        2. Run the tests.
        3. If a test fails, investigate why.
        4. Fix the bug in the source code.
        5. Run the tests again.
        6. Give me a concise summary of:
            - what was wrong
            - what you changed
            - the final test result
        """
    }
  ]
})

#### Expected final state

The agent should change:
```python
def add(a, b):
    return a - b
```
To :  
```python
def add(a, b):
    return a + b
```

And the final test run should report something equivalent to:  2 passed


In [8]:
print_messages(response)

HumanMessage: You are working on a Python project.
        Workspace: 
/Users/geetikabatra/genai_bootcamp/implementation/langchain-learning/notebooks/07_middlewares/shell_agent_workspace
        Please:
        1. Inspect the files.
        2. Run the tests.
        3. If a test fails, investigate why.
        4. Fix the bug in the source code.
        5. Run the tests again.
        6. Give me a concise summary of:
            - what was wrong
            - what you changed
            - the final test result

================================================================================

Agent: AIMessage with tool calls:

Tool Call 1 Name: shell
Tool Call 1 Args: {'command': "pwd && ls -la && find . -maxdepth 3 -type f | sed 's#^./##' | sort | head -200", 
'restart': True}

================================================================================

...

================================================================================

ToolMessage: ..                                                                       [100%]
2 passed in 0.00s

================================================================================

Agent: Summary:
- **What was wrong:** `calculator.py` had a bug in `add(a, b)`: it returned `a - b` instead of `a + b`.
- **What I changed:** Updated `add` to return the sum: `return a + b`.
- **Final test result:** Ran `pytest -q` again; all tests pass: **2 passed**.

================================================================================